# VoxSync Colab

Pipeline:
1. Transcribir (Whisper)
2. Traducir a espanol (MarianMT local)
3. Detectar genero/edad por segmento (pitch)
4. Doblar con voces ES (mujer / hombre / mayor)
5. Video final con audio doblado + subtitulos opcionales

**Entorno > GPU (T4)**


## 1. Instalar


In [ ]:
!pip install -q openai-whisper yt-dlp "transformers>=4.40,<5" sentencepiece sacremoses edge-tts librosa soundfile
!apt -y -qq install ffmpeg
print("OK")


## 2. Config


In [ ]:
YOUTUBE_URL = ""  # o link de YouTube
SOURCE_LANG = "en"
TARGET_LANG = "es"
WHISPER_MODEL = "small"
BURN_SUBTITLES = True
DUB_AUDIO = True  # True = reemplaza voces por TTS segun genero/edad

VOICE_FEMALE = "es-ES-ElviraNeural"
VOICE_MALE = "es-ES-AlvaroNeural"
VOICE_SENIOR_M = "es-ES-AlvaroNeural"
VOICE_SENIOR_F = "es-ES-ElviraNeural"
VOICE_CHILD = "es-ES-ElviraNeural"
print("Config OK")


## 3. Cargar video


In [ ]:
from pathlib import Path
import subprocess

work = Path("/content/voxsync")
work.mkdir(exist_ok=True)

if YOUTUBE_URL.strip():
    print("Descargando YouTube...")
    subprocess.run([
        "yt-dlp", "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b",
        "-o", str(work / "%(title)s.%(ext)s"), "--no-playlist", YOUTUBE_URL.strip()
    ], check=True)
    cands = []
    for ext in (".mp4", ".webm", ".mkv"):
        cands.extend(work.glob("*" + ext))
    video_path = max(cands, key=lambda p: p.stat().st_mtime)
else:
    from google.colab import files
    print("Sube un video...")
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    video_path = work / name
    video_path.write_bytes(uploaded[name])

print("Video:", video_path)


## 4. Extraer audio + transcribir


In [ ]:
import whisper

audio_path = work / "audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", str(video_path),
    "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(audio_path)
], check=True, capture_output=True)

print("Transcribiendo...")
model = whisper.load_model(WHISPER_MODEL)
result = model.transcribe(str(audio_path), language=SOURCE_LANG or None, verbose=False)
segments = result["segments"]
print(len(segments), "segmentos")


## 5. Traducir (MarianMT)


In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-es"
if SOURCE_LANG == "es" and TARGET_LANG == "en":
    model_name = "Helsinki-NLP/opus-mt-es-en"

print("Cargando", model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)
mt_model = MarianMTModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
mt_model = mt_model.to(device).eval()

def translate_batch(texts, max_length=256):
    texts = [t if (t and t.strip()) else "." for t in texts]
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = mt_model.generate(**inputs, max_length=max_length)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

texts = [(s.get("text") or "").strip() for s in segments]
translations = []
if SOURCE_LANG == TARGET_LANG:
    translations = list(texts)
else:
    for i in range(0, len(texts), 16):
        batch = texts[i:i+16]
        out = translate_batch(batch)
        for j, t in enumerate(batch):
            translations.append(out[j] if t else "")
        print(f"  trad {min(i+16, len(texts))}/{len(texts)}")

for seg, tr in zip(segments, translations):
    seg["translation"] = tr
print("Traduccion OK")


## 6. Detectar genero / edad por segmento (pitch)


In [ ]:
import numpy as np
import librosa
from collections import Counter

y, sr = librosa.load(str(audio_path), sr=16000)

def classify_voice(start, end):
    i0 = max(0, int(start * sr))
    i1 = min(len(y), int(end * sr))
    if i1 - i0 < sr * 0.15:
        return "adult_male", 0.3
    chunk = y[i0:i1]
    try:
        f0, _, _ = librosa.pyin(chunk, fmin=60, fmax=400, sr=sr)
        f0_clean = f0[~np.isnan(f0)] if f0 is not None else np.array([])
    except Exception:
        f0_clean = np.array([])
    if len(f0_clean) < 3:
        return "adult_male", 0.3
    med = float(np.median(f0_clean))
    if med >= 220:
        return "child", 0.7
    if med >= 170:
        return "adult_female", 0.75
    if med >= 140:
        return "adult_female", 0.55
    if med >= 100:
        return "adult_male", 0.75
    return "senior_male", 0.7

for i, seg in enumerate(segments):
    label, conf = classify_voice(seg["start"], seg["end"])
    seg["voice_type"] = label
    seg["voice_conf"] = conf
    if (i + 1) % 50 == 0:
        print(f"  atributos {i+1}/{len(segments)}")

print("Distribucion:", Counter(s["voice_type"] for s in segments))


## 7. Doblar con Edge-TTS (mujer / hombre / mayor)


In [ ]:
import asyncio
import edge_tts
import soundfile as sf

def pick_voice(voice_type):
    if voice_type == "adult_female":
        return VOICE_FEMALE, "+0Hz", "+0%"
    if voice_type == "child":
        return VOICE_CHILD, "+15Hz", "+10%"
    if voice_type == "senior_male":
        return VOICE_SENIOR_M, "-25Hz", "-8%"
    if voice_type == "senior_female":
        return VOICE_SENIOR_F, "-15Hz", "-5%"
    return VOICE_MALE, "+0Hz", "+0%"

async def synth_one(text, out_path, voice, rate, pitch):
    text = (text or "").strip()
    if not text:
        sf.write(str(out_path), np.zeros(1600, dtype=np.float32), 16000)
        return
    communicate = edge_tts.Communicate(text, voice=voice, rate=rate, pitch=pitch)
    await communicate.save(str(out_path))

async def synth_all(segments, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, seg in enumerate(segments):
        voice, pitch, rate = pick_voice(seg.get("voice_type", "adult_male"))
        path = out_dir / f"seg_{i:04d}.mp3"
        await synth_one(seg.get("translation") or seg.get("text"), path, voice, rate, pitch)
        seg["tts_path"] = str(path)
        if (i + 1) % 20 == 0:
            print(f"  TTS {i+1}/{len(segments)}")

if DUB_AUDIO:
    print("Generando voces (Edge-TTS)...")
    await synth_all(segments, work / "tts_segs")
    print("TTS OK")
else:
    print("DUB_AUDIO=False")


## 8. Armar pista de audio doblada


In [ ]:
import json as _json

def load_audio_any(path, target_sr=16000):
    path = Path(path)
    if path.suffix.lower() == ".mp3":
        tmp = path.with_suffix(".wav")
        subprocess.run([
            "ffmpeg", "-y", "-i", str(path), "-ar", str(target_sr), "-ac", "1", str(tmp)
        ], check=True, capture_output=True)
        data, _sr = sf.read(str(tmp))
    else:
        data, _sr = librosa.load(str(path), sr=target_sr, mono=True)
    return np.asarray(data, dtype=np.float32), target_sr

dub_path = None
if DUB_AUDIO:
    probe = subprocess.run([
        "ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(video_path)
    ], capture_output=True, text=True, check=True)
    duration = float(_json.loads(probe.stdout)["format"]["duration"])
    sr = 16000
    total = int(duration * sr) + sr
    mix = np.zeros(total, dtype=np.float32)

    for i, seg in enumerate(segments):
        tts = seg.get("tts_path")
        if not tts or not Path(tts).exists():
            continue
        data, _ = load_audio_any(tts, sr)
        start = int(seg["start"] * sr)
        if start >= total:
            continue
        slot = int((seg["end"] - seg["start"]) * sr)
        if slot > 0 and len(data) > int(slot * 1.35):
            data = data[: int(slot * 1.35)]
        end = min(start + len(data), total)
        data = data[: end - start]
        mix[start:end] += data
        if (i + 1) % 50 == 0:
            print(f"  mezclando {i+1}/{len(segments)}")

    peak = np.max(np.abs(mix)) + 1e-8
    if peak > 0.95:
        mix = mix * (0.95 / peak)

    dub_path = work / "audio_doblado.wav"
    sf.write(str(dub_path), mix, sr)
    print("Audio doblado:", dub_path)


## 9. SRT + video final


In [ ]:
def ts(seconds):
    total = int(float(seconds))
    ms = int((float(seconds) - total) * 1000)
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

base = video_path.stem
srt_path = work / f"{base}.es.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        text = (seg.get("translation") or seg["text"]).strip()
        f.write(f"{i}\n{ts(seg['start'])} --> {ts(seg['end'])}\n{text}\n\n")
print("SRT:", srt_path)

out_video = work / f"{base}_ES_DOBLADO.mp4"
audio_input = str(dub_path) if dub_path else str(audio_path)

if BURN_SUBTITLES:
    srt_esc = str(srt_path).replace("\\", "/").replace(":", "\\:")
    vf = f"subtitles={srt_esc}:force_style='FontSize=18,Outline=1'"
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-i", audio_input,
        "-map", "0:v:0", "-map", "1:a:0",
        "-vf", vf,
        "-c:v", "libx264", "-preset", "fast", "-crf", "23",
        "-c:a", "aac", "-shortest",
        str(out_video)
    ]
else:
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-i", audio_input,
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac", "-shortest",
        str(out_video)
    ]

print("Renderizando video final...")
subprocess.run(cmd, check=True, capture_output=True)
print("Video:", out_video)


## 10. Descargar


In [ ]:
from google.colab import files
files.download(str(srt_path))
files.download(str(out_video))
if dub_path:
    files.download(str(dub_path))
print("Listo: video con voces sinteticas (mujer/hombre/mayor) + subtitulos")
